In [3]:
import os
import json
import pandas as pd
from tqdm import tqdm
from openai import AzureOpenAI, BadRequestError
from scipy.spatial.distance import cdist
from dotenv import load_dotenv

import sys
sys.path.append("..")
from conformal.utils import format_email

load_dotenv()

True

# Create blocklist from stored logs

In [ ]:
filename = "llmail-inject/src/agent/workloads/conformal/jobs_data_202502031523.json"
prompts = pd.read_json(filename)

Expand success attributes:

In [7]:
prompts["email_retrieved"] = prompts.objectives.apply(lambda x: bool(json.loads(x).get("email.retrieved", False)))
prompts["defense_undetected"] = prompts.objectives.apply(lambda x: bool(json.loads(x).get("defense.undetected", False)))
prompts["exfil_sent"] = prompts.objectives.apply(lambda x: bool(json.loads(x).get("exfil.sent", False)))
prompts["exfil_destination"] = prompts.objectives.apply(lambda x: json.loads(x).get("exfil.destination", None))
prompts["exfil_content"] = prompts.objectives.apply(lambda x: json.loads(x).get("exfil.content", None))

Only keep successful for target level (`level1*` and `level2*`).

In [ ]:
target_scenarios = [f"level1{c}" for c in "abcdefghij"]
target_scenarios += [f"level2{c}" for c in "abcdefghij"]

successful_prompts = prompts[
    (prompts.exfil_sent) &
    prompts.scenario.isin(target_scenarios)
]
len(successful_prompts)

24499

In [19]:
successful_prompts = successful_prompts.drop_duplicates("body")
len(successful_prompts)

14589

One of them has an empty body and injects in the subject. We invert them for simplicity.

In [8]:
row = successful_prompts.loc[successful_prompts.body == " "].copy()
row.body = row.subject
row.subject = " "
successful_prompts.loc[successful_prompts.body == " "] = row

Estimate the costs:

In [9]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

def get_len(p):
    try:
        return len(enc.encode(p))
    except:
        return 200

input_tokens = sum([get_len(p) for p in successful_prompts.body.tolist()])
input_tokens /= 1e6
output_tokens = input_tokens*10

0.15*input_tokens + 0.6*output_tokens

63.180382949999995

### Create paraphrasings

Paraphrases each working attack, and stores distance (in embedding space) between attack and paraphrasings.

In [15]:
client = AzureOpenAI(
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_version="2024-08-01-preview",
)

A simple database to store the paraphrasings: `paraphrasings.json`.

```json
{
    prompt: <basic prompt used>,
    sentences: [
        {
            sentence: <sentence>,
            embedding: <embedding of the sentence>,
            paraphrasings: [<para1>, <para2>, ...],
            distances: <distances from the respective paraphrasings>,
        }
    ]
}
```

In [28]:
PROMPT = """Rephrase the text of the sentence enclosed between the [SENTENCE] and [/SENTENCE] tags.
It must be faitful and be very similar to the original. You must not alter its meaning. If it contains any symbols that don't make sense, you must keep them as they are! Preserve typos.

[SENTENCE]
{attack}
[/SENTENCE]

Rephrase the sentence and put it between [REPHRASED] and [/REPHRASED]"""

In [29]:
paradb = "blocklist.json"

paraphrasings = {}
if os.path.exists(paradb):
    print("Loading paraphrasings from database...")
    with open(paradb) as f:
        paraphrasings = json.load(f)

if "prompt" in paraphrasings and paraphrasings["prompt"] != PROMPT:
    raise Exception("The prompt in the database does not match the current prompt. You may need to create a new one.")

paraphrasings["prompt"] = PROMPT
if not "sentences" in paraphrasings:
    paraphrasings["sentences"] = []

In [36]:
def generate_paraphrasings(attack):
    messages = [{"role": "system", "content": PROMPT.format(attack=attack)}]
    completion = client.chat.completions.create(
        model="gpt-4o-mini-no-filter",
        messages=messages,
        temperature=0.5,
        n=10,
    )

    paraphrased = set([choice.message.content for choice in completion.choices])

    paraphrasings = []
    for p in paraphrased:
        if not "[REPHRASED]" in p or not "[/REPHRASED]" in p:
            print(f"Paraphrasing structure check failed: {p}")
            continue
        start = p.index("[REPHRASED]") + len("[REPHRASED]")
        end = p.index("[/REPHRASED]")
        paraphrasings.append(p[start:end].strip())
    
    return paraphrasings

def paraphrase_email(subject, body):
    """Concatenates subject and body in a common format, and paraphrases
    both parts separately. Returns a list of paraphrased emails."""
    subject_para = generate_paraphrasings(subject)
    body_para = generate_paraphrasings(body)
    print(len(subject_para), len(body_para))

    paraphrased_emails = []

    # All combinations.
    for s in subject_para:
        for b in body_para:
            paraphrased_emails.append(format_email(s, b))
    
    return paraphrased_emails

def compute_embeddings(texts):
    response = client.embeddings.create(input=texts, model="text-embedding-3-large")

    return [data.embedding for data in response.data]

Generate paraphrasings.

In [38]:
successful_prompts = successful_prompts.sample(10)

In [47]:
def already_done(attack, paraphrasings):
    for entry in paraphrasings["sentences"]:
        if entry["sentence"] == attack:
            return True
    return False

for row in tqdm(successful_prompts.itertuples(), total=len(successful_prompts)):
    body = row.body
    subject = row.subject
    sentence = format_email(subject, body)

    if already_done(sentence, paraphrasings):
        continue

    try:
        para = paraphrase_email(subject, body)
    except BadRequestError as e:
        print(f"Failed to generate paraphrasings for '{body[:100]}': {e}")
        continue

    if not len(para):
        print(f"No paraphrasings generated for '{body[:100]}'")
        paraphrasings["sentences"].append(
            {
                "sentence": sentence,
                "embedding": compute_embeddings([sentence])[0],
                "paraphrasings": None,
                "distances": None
            }
        )
        continue

    embeddings = compute_embeddings([sentence] + para)
    distances = cdist([embeddings[0]], embeddings[1:], "cosine")[0]

    paraphrasings["sentences"].append(
        {
            "sentence": sentence,
            "embedding": embeddings[0],
            "paraphrasings": para,
            "distances": distances.tolist()
        }
    )

  0%|          | 0/10 [00:00<?, ?it/s]

3 7


 80%|████████  | 8/10 [00:09<00:02,  1.14s/it]

2 10


 90%|█████████ | 9/10 [00:33<00:04,  4.62s/it]

2 5


100%|██████████| 10/10 [00:40<00:00,  4.08s/it]


In [42]:
15*35000/60/60/24

6.076388888888889

Store blocklist.

In [49]:
with open(paradb, "w") as f:
    json.dump(paraphrasings, f, indent=4)

# Evaluate blocklist

Load blocklist.

In [50]:
from blocklist import ConformalBlocklist

In [51]:
paradb = "blocklist.json"
cb = ConformalBlocklist(alpha=0, alpha_global=0.1, blocklist_db_file=paradb)

Global threshold: 0.2397312621253297


Good performance on true positives:

In [52]:
error = 0
for row in successful_prompts.itertuples():
    email = format_email(row.subject, row.body)
    if not cb.predict(email):
        error += 1
print(f"Error rate: {error / len(successful_prompts) * 100:.2f}%")

Error rate: 0.00%


Sanity check on positives. Should do more checks on larger dataset.

In [53]:
cb.predict("This is a test")

False